# Colab 14 KI Beratungsteam

**Auftrag · 50 Minuten insgesamt · Gruppen mit 3 bis 4 Personen**

Ein Sportgeschäft möchte Kleidungsbilder automatisch erkennen lassen. Ihr seid das Beratungsteam. Untersucht das Netz und empfehlt eine Lernrate und eine Netzbreite. Belegt euren Rat mit Ergebnissen.

Öffnet **Colab_14_Gruppenauftrag.pdf** und gemeinsam **Colab_14_Beratungsbericht.docx**.

**Startet eure Stoppuhr.** Bis Minute 6: Auftrag lesen, Rollen verteilen und Bericht öffnen. Nutzt ein laufendes Colab pro Gruppe. Nach E1, E2, E3 und E4 wechseln die Rollen im Uhrzeigersinn. Zu dritt werden Vorhersage und Erklärung zusammengelegt.

| Rolle | Eure Aufgabe |
|---|---|
| Bedienung | Zellen ausführen; nur die zwei erlaubten Zahlen ändern |
| Vorhersage | Jedes Mitglied zuerst nach seiner Vermutung fragen |
| Datenprüfung | Werte und Auswahlregel prüfen |
| Erklärung und Abgabe | Gruppenentscheid im gemeinsamen Bericht festhalten |

**Euer Ziel:** Fünf belegte Missionen ergeben eure gemeinsame Beraterfreigabe. Auch eine widerlegte Vermutung kann gut erklärt sein.

**Technische Reserve:** Hängt die Technik länger als drei Minuten oder ist eine Phase bis zu ihrer Zeitgrenze nicht fertig, wechselt vollständig zu **Colab_14_Ersatzdaten.pdf**. Dieselben Beispielwerte stehen auch unten im Notebook. Markiert im Bericht «Ersatzdaten» und ersetzt bereits notierte Messwerte. Eure ursprüngliche Vermutung bleibt stehen. Die letzten fünf Minuten sind für die Abgabe reserviert.

## Bis Minute 13 Start A bis C und E1

1. Stellt die Verbindung zum Notebook her. Unter **Laufzeit → Laufzeittyp ändern** könnt ihr eine **GPU**, zum Beispiel T4, wählen.
2. Falls keine GPU frei ist: Nutzt die CPU. Sie genügt für die Aufgabe. Die GPU kann viele Rechnungen gleichzeitig ausführen; sie garantiert keine besseren Vorhersagen.
3. Führt **Start A**, **Start B** und **Start C** nacheinander aus. Wartet auf die Ausgabe. Start A zeigt das tatsächlich verwendete Gerät.
4. Danach folgt E1. Während des Rechnens lest ihr die Begriffshilfe und besprecht die Aufgabe.

Beim Wechsel des Laufzeittyps startet die Umgebung neu. Führt dann alle Startzellen erneut aus. Downloadfehler? Start B einmal erneut versuchen. Nach drei Minuten ohne Fortschritt nutzt ihr den Ersatzweg.

In [ ]:
#@title Start A – Rechengerät prüfen { display-mode: "form" }
import copy
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))

print("PyTorch:", torch.__version__)
if device.type == "cuda":
    print("GPU aktiv:", torch.cuda.get_device_name(0))
else:
    print("CPU aktiv. Es ist keine CUDA-GPU verfügbar. Das Colab funktioniert trotzdem.")

# Eine kleine echte Rechnung auf dem gewählten Gerät.
probe = torch.tensor([2.0, 3.0], device=device)
print("Gerät der Proberechnung:", probe.device)
print("Proberechnung 2 + 3 =", probe.sum().item())

class_names = [
    "T-Shirt/Oberteil", "Hose", "Pullover", "Kleid", "Mantel",
    "Sandale", "Hemd (Shirt)", "Turnschuh", "Tasche", "Stiefelette",
]


In [ ]:
#@title Start B – Feste Bildmengen laden { display-mode: "form" }
transform = transforms.Compose([transforms.ToTensor()])
data_root = Path("fashion_mnist_data")

# HTTPS-Quelle des ursprünglichen Fashion-MNIST-Projekts.
# Die üblichen torchvision-Quellen bleiben als weitere Versuche erhalten.
secure_mirror = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
if secure_mirror not in datasets.FashionMNIST.mirrors:
    datasets.FashionMNIST.mirrors = [secure_mirror] + datasets.FashionMNIST.mirrors

try:
    full_train_dataset = datasets.FashionMNIST(
        root=str(data_root), train=True, download=True, transform=transform,
    )
except (RuntimeError, OSError) as error:
    raise RuntimeError(
        "Die Bilder konnten nicht geladen werden. Prüfe die Internetverbindung "
        "und starte nur diese Zelle erneut. Bei weiterem Fehler hilft die Lehrperson."
    ) from error

# Alle drei Mengen kommen hier nur aus dem offiziellen Trainingsbestand.
train_subset, validation_subset, unused_subset = random_split(
    full_train_dataset, [6000, 2000, len(full_train_dataset) - 8000],
    generator=torch.Generator().manual_seed(42),
)
demo_subset = Subset(train_subset, range(512))
batch_size = 64
comparison_epochs = 6
run_seed = 7
learning_rate_runs = {}
width_runs = {}
test_result = None

print(f"\nTraining für Vergleiche: {len(train_subset):,} Bilder".replace(",", "'"))
print(f"Validierung: {len(validation_subset):,} andere Bilder".replace(",", "'"))
print(f"Sonderversuch Überanpassung: nur {len(demo_subset)} Trainingsbilder")
print("Test: 10'000 getrennte Bilder. Wir werten sie erst in E5 aus.")
print("Die übrigen 52'000 Trainingsbilder verwenden wir aus Zeitgründen nicht.")


### Kurze Begriffshilfe ohne Tafel-Einstieg

- **Training:** Mit diesen Bildern verändert das Netz seine Gewichte und Bias-Werte.
- **Validierung:** Mit anderen Bildern vergleichen wir Einstellungen und wählen einen Modellzustand.
- **Test:** Diese Bilder bleiben bis E5 zurückgehalten. Dann prüfen wir die vorher festgelegte Auswahl.
- **Parameter:** Die veränderbaren Zahlenwerte des Netzes: Gewichte und Bias-Werte.

Normalerweise arbeiten wir mit **6'000 Trainingsbildern und 2'000 Validierungsbildern**. Nur E3 nutzt **512 Trainingsbilder**. Die **10'000 Testbilder** gehören zu einem getrennten Datensatzteil.

Ihr ändert später nur **chosen_learning_rate** in E4 und **chosen_hidden_nodes** in E5. Der übrige Python-Code bleibt unverändert.

In [ ]:
#@title Start C – Modell und Hilfsfunktionen bereitstellen { display-mode: "form" }
# Diesen vorbereiteten Code nicht ändern.
class FashionMNISTModel(nn.Module):
    def __init__(self, hidden_nodes=128):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, hidden_nodes),
            nn.ReLU(),
            nn.Linear(hidden_nodes, 10),
            nn.LogSoftmax(dim=1),
        )

    def forward(self, images):
        pixels = self.flatten(images)
        log_probabilities = self.linear_relu_stack(pixels)
        return log_probabilities


loss_function = nn.NLLLoss()


def make_loader(dataset, shuffle=False, seed=run_seed):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0, pin_memory=(device.type == "cuda"),
    )


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device, non_blocking=(device.type == "cuda"))
            labels = labels.to(device, non_blocking=(device.type == "cuda"))
            log_probabilities = model(images)
            loss = loss_function(log_probabilities, labels)
            total_loss += loss.item() * len(labels)
            correct += (log_probabilities.argmax(dim=1) == labels).sum().item()
            total += len(labels)
    return {"loss": total_loss / total, "accuracy": 100 * correct / total,
            "correct": int(correct), "total": total}


def copy_state(model):
    # Eine unabhängige Kopie bleibt erhalten, auch wenn das Modell weiterlernt.
    return {name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()}


def run_experiment(name, learning_rate, hidden_nodes, epochs,
                   training_data=train_subset, seed=run_seed):
    if test_result is not None:
        raise RuntimeError("Der Schlusstest ist bereits erfolgt. Vergleiche jetzt die Ergebnisse; optimiere nicht am Test weiter.")
    if learning_rate <= 0 or not math.isfinite(learning_rate):
        raise ValueError("Die Lernrate muss eine positive Zahl sein.")
    set_seed(seed)
    model = FashionMNISTModel(hidden_nodes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    train_loader = make_loader(training_data, shuffle=True, seed=seed)
    train_check_loader = make_loader(training_data)
    validation_loader = make_loader(validation_subset)
    history = []
    best_loss = float("inf")
    best_state = None
    first_state = None
    best_epoch = None
    if device.type == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    print(f"{name}: Lernrate {learning_rate:g}, {hidden_nodes} versteckte Neuronen, {epochs} Epochen")
    print("Gerät des Modells:", next(model.parameters()).device)
    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in train_loader:
            images = images.to(device, non_blocking=(device.type == "cuda"))
            labels = labels.to(device, non_blocking=(device.type == "cuda"))
            log_probabilities = model(images)
            loss = loss_function(log_probabilities, labels)
            if not torch.isfinite(loss):
                raise RuntimeError("Der Loss ist keine endliche Zahl mehr. Wähle eine kleinere Lernrate und starte den Versuch neu.")
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        # Beide Mengen werden mit dem Modellzustand am Epochenende geprüft.
        train_metrics = evaluate(model, train_check_loader)
        validation_metrics = evaluate(model, validation_loader)
        history.append({
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "validation_loss": validation_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "validation_accuracy": validation_metrics["accuracy"],
        })
        if epoch == 1:
            first_state = copy_state(model)
        if validation_metrics["loss"] < best_loss:
            best_loss = validation_metrics["loss"]
            best_epoch = epoch
            best_state = copy_state(model)
        if epochs <= 6 or epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"Epoche {epoch:2d}: Loss Training {train_metrics['loss']:.3f}, "
                  f"Validierung {validation_metrics['loss']:.3f}; "
                  f"Treffer Training {train_metrics['accuracy']:.1f} %, "
                  f"Validierung {validation_metrics['accuracy']:.1f} %")
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    return {
        "name": name, "learning_rate": learning_rate, "hidden_nodes": hidden_nodes,
        "epochs": epochs, "training_count": len(training_data), "seed": seed,
        "history": history, "best_epoch": best_epoch, "best_state": best_state,
        "first_state": first_state, "final_state": copy_state(model),
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "seconds": elapsed,
    }


def show_history(run):
    frame = pd.DataFrame(run["history"])
    display(frame.rename(columns={
        "epoch": "Epoche", "train_loss": "Loss Training", "validation_loss": "Loss Validierung",
        "train_accuracy": "Treffer Training %", "validation_accuracy": "Treffer Validierung %",
    }).round(3))
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(frame["epoch"], frame["train_loss"], label="Training", marker=".")
    axes[0].plot(frame["epoch"], frame["validation_loss"], label="Validierung", marker=".")
    axes[0].set_ylabel("Loss (keine Prozentzahl)")
    axes[0].set_title("Loss: kleiner ist besser")
    axes[1].plot(frame["epoch"], frame["train_accuracy"], label="Training", marker=".")
    axes[1].plot(frame["epoch"], frame["validation_accuracy"], label="Validierung", marker=".")
    axes[1].set_ylabel("Trefferquote in %")
    axes[1].set_title("Trefferquote: höher ist besser")
    axes[1].set_ylim(0, 100)
    for axis in axes:
        axis.set_xlabel("Epoche")
        axis.legend()
        axis.grid(alpha=0.25)
    fig.suptitle(run["name"])
    fig.tight_layout()
    plt.show()


def compare_runs(runs, setting):
    rows = []
    for run in runs:
        last = run["history"][-1]
        rows.append({
            setting: run["learning_rate"] if setting == "Lernrate" else run["hidden_nodes"],
            "Epochen": run["epochs"], "Loss Validierung nach 6": last["validation_loss"],
            "Treffer Validierung %": last["validation_accuracy"],
            "Parameter": run["parameter_count"], "Zeit Sekunden": run["seconds"],
        })
    frame = pd.DataFrame(rows)
    display_frame = frame.copy()
    if setting == "Lernrate":
        # Nicht auf 0.000 runden: 0.0001 muss lesbar bleiben.
        display_frame[setting] = display_frame[setting].map(lambda value: f"{value:g}")
    display(display_frame.round(3))
    fig, axis = plt.subplots(figsize=(6, 3.5))
    axis.bar([str(value) for value in frame[setting]], frame["Loss Validierung nach 6"])
    axis.set_xlabel(setting)
    axis.set_ylabel("Loss Validierung nach 6 Epochen")
    axis.set_title("Jeweils sechs Epochen: kleinerer Loss ist besser")
    fig.tight_layout()
    plt.show()
    return frame


print("Bereit: Grundmodell 784 → 128 → 10, ReLU, LogSoftmax, NLLLoss und Adam.")
print("Ein Vergleichslauf startet das Modell und den Optimizer frisch.")


## Mission 1 Training verstehen bis Minute 13

**Eine Epoche** ist ein vollständiger Durchgang durch alle Trainingsbilder. Bei sechs Epochen werden dieselben Bilder sechsmal verwendet, jeweils neu gemischt. Ein **Batch** ist ein kleines Paket von Bildern; hier sind es bis zu 64 Bilder.

Der **Loss** bewertet, wie schlecht Vorhersagen zu den richtigen Antworten passen. **Kleiner ist besser. Er ist keine Fehlerquote in Prozent.** Die Trefferquote zählt dagegen den Anteil richtig eingeordneter Bilder.

Führt E1 aus. Vergleicht Epoche 1 und 6: Was passiert mit Trainings-Loss, Validierungs-Loss und Validierungsquote?

**Gemeinsam lösen:** Eine Person erklärt die Epoche und eine andere den Loss. Notiert beide Erklärungen mit Namen kurz im Bericht. Wechselt danach die Rollen.

In [ ]:
#@title E1 – Dasselbe Netz sechs Epochen trainieren { display-mode: "form" }
baseline_run = run_experiment("E1 Grundmodell", 0.001, 128, comparison_epochs)
show_history(baseline_run)


## Mission 2 Die Lernrate wählen bis Minute 23

Beim Einparken können grosse Lenkkorrekturen am Ziel vorbeiführen. Kleine Korrekturen brauchen länger. Die **Lernrate** beeinflusst, wie stark Gewichte verändert werden. Der Optimizer **Adam** berechnet die einzelnen Änderungen; die Lernrate ist eine Einstellung dafür.

**Vor dem Ausführen:** Jedes Mitglied nennt einen Tipp: Welche Lernrate wird nach sechs Epochen den kleinsten Validierungs-Loss haben: **0.0001, 0.001 oder 0.01**? Notiert eine gemeinsame Vermutung mit Grund im Bericht.

Führt E2 aus. Jeder neue Vergleichslauf startet ein frisches Netz und einen frischen Optimizer. Bei gleicher Netzform bleiben Anfangsgewichte, Bilder, Reihenfolge, Batchgrösse und sechs Epochen gleich. Nur die Lernrate ändert sich. Der Lauf 0.001 aus E1 wird wiederverwendet.

**Eure Auswahlregel:** Wählt den **kleinsten Validierungs-Loss nach sechs Epochen**. Übertragt die drei Loss-Werte. Erklärt kurz, ob eure Vermutung passt. Haltet eure Lernrate für E4 bereit. Wechselt danach die Rollen.

In [ ]:
#@title E2 – Drei Lernraten fair vergleichen { display-mode: "form" }
# 0.001 wurde in E1 schon unter genau diesen Bedingungen trainiert.
learning_rate_runs = {0.001: baseline_run}
for learning_rate in [0.0001, 0.001, 0.01]:
    if learning_rate != 0.001:
        learning_rate_runs[learning_rate] = run_experiment(
            f"E2 Lernrate {learning_rate:g}", learning_rate, 128, comparison_epochs,
        )
print("Die Zeile 0.001 verwendet den vorhandenen Lauf E1; sie trainiert ihn nicht weiter.")
learning_rate_table = compare_runs(
    [learning_rate_runs[value] for value in [0.0001, 0.001, 0.01]], "Lernrate",
)
best_learning_rate = min(
    learning_rate_runs,
    key=lambda value: learning_rate_runs[value]["history"][-1]["validation_loss"],
)
print("Kleinster Validierungs-Loss nach sechs Epochen bei Lernrate:", best_learning_rate)


## Mission 3 Den Kurvenfall lösen bis Minute 32

**Euer Fall:** Ein Netz wird mit den Übungsbildern immer besser. Trotzdem kann es bei anderen Bildern schlechter werden. Könnt ihr das in den Kurven erkennen?

Für E3 ändern wir die Bedingungen bewusst: **512 Trainingsbilder, 256 versteckte Neuronen, Lernrate 0.003 und 50 Epochen**. Die Validierungsbilder bleiben getrennt. E3 ist ein Sonderversuch und kein fairer direkter Qualitätsvergleich mit E1.

1. Führt E3 aus. Das Netz trainiert zur Beobachtung bis Epoche 50.
2. Vergleicht den gespeicherten Zustand mit dem **kleinsten Validierungs-Loss** mit Epoche 50.
3. Übertragt beide Zustände in den Bericht. Welchen würdet ihr für andere Bilder retten? Begründet mit **beiden Kurven über mehrere Epochen**.

Ein weiter sinkender Trainings-Loss bei längerfristig steigendem Validierungs-Loss spricht für **Überanpassung**. Ein einzelner Ausschlag genügt nicht. Bei unklarem Verlauf ist «Überanpassung ist hier nicht deutlich sichtbar» ein gültiges Urteil mit Beleg.

Der beste Zustand bleibt als Kopie gespeichert. Das Training wird hier nicht vorzeitig beendet. Wechselt danach die Rollen.

In [ ]:
#@title E3 – Überanpassung unter der Lupe { display-mode: "form" }
# Bewusster Sonderversuch: weniger Bilder und mehr Epochen.
# Nicht als fairen Qualitätsvergleich mit E1 oder E2 verwenden.
overfit_run = run_experiment(
    "E3 Überanpassung unter der Lupe", learning_rate=0.003,
    hidden_nodes=256, epochs=50, training_data=demo_subset,
)
show_history(overfit_run)
best_row = overfit_run["history"][overfit_run["best_epoch"] - 1]
last_row = overfit_run["history"][-1]
print("Kleinster Validierungs-Loss bei Epoche:", overfit_run["best_epoch"])
display(pd.DataFrame([
    {"Zustand": "Kleinster Validierungs-Loss", **best_row},
    {"Zustand": "Letzte Epoche", **last_row},
]).rename(columns={
    "epoch": "Epoche", "train_loss": "Loss Training", "validation_loss": "Loss Validierung",
    "train_accuracy": "Treffer Training %", "validation_accuracy": "Treffer Validierung %",
}).round(3))
print("Der beste Modellzustand ist als eigene Kopie gespeichert.")
print("Wir haben zur Beobachtung trotzdem bis Epoche 50 weitertrainiert.")


## Mission 4 Die Netzbreite wählen bis Minute 41

Zurück zu **6'000 Trainingsbildern und sechs Epochen**. Die **Netzbreite** ist hier die Zahl der Neuronen in der versteckten Schicht. Wir vergleichen **16, 128 und 256**. Mehr Neuronen bedeuten mehr Parameter und Rechenarbeit. Ob sie helfen, müssen wir prüfen.

1. Tragt bei **chosen_learning_rate** eure Lernrate aus E2 ein. Ändert nur diese Zahl.
2. Führt E4 aus. Übertragt die drei Validierungs-Loss-Werte und die Parameterzahlen in den Bericht.
3. Wählt die Breite mit dem **kleinsten Validierungs-Loss nach sechs Epochen**.
4. Haltet **Lernrate und Netzbreite vor E5** im Bericht fest. Nennt auch einen Nachteil eines grösseren Netzes.

Gleiche Daten, Lernrate und Epochenzahl erlauben einen sinnvollen Vergleich. Unterschiedliche Netzbreiten können aber nicht dieselben Anfangsgewichte haben, weil sie unterschiedlich viele Gewichte besitzen. Ein einzelner Lauf beweist keinen allgemeinen Vorteil.

Wechselt danach die Rollen. Eure Auswahl steht jetzt fest.

In [ ]:
#@title E4 – Lernrate auswählen und Netzbreiten vergleichen
# Trage hier deine Wahl aus E2 ein. Nur diese Zahl ändern.
chosen_learning_rate = 0.001 #@param {type:"number"}

if chosen_learning_rate not in learning_rate_runs:
    raise ValueError("Wähle eine der drei geprüften Lernraten: 0.0001, 0.001 oder 0.01.")
if test_result is not None:
    raise RuntimeError("Der Schlusstest ist bereits erfolgt. Optimiere nicht am Test weiter.")
width_runs = {128: learning_rate_runs[chosen_learning_rate]}
for hidden_nodes in [16, 256]:
    width_runs[hidden_nodes] = run_experiment(
        f"E4 Netzbreite {hidden_nodes}", chosen_learning_rate,
        hidden_nodes, comparison_epochs,
    )
print("Die Breite 128 verwendet den vorhandenen Lauf aus E2.")
width_table = compare_runs([width_runs[value] for value in [16, 128, 256]], "Versteckte Neuronen")
print("Gleiche Bilder, sechs Epochen und gleiche Lernrate; nur die Netzbreite ändert sich.")
print("Verschiedene Netzbreiten können nicht dieselben Anfangsgewichte haben.")

## Mission 5 Den Auftrag abschliessen bis Minute 45

**Prüft zuerst:** Stehen Lernrate und Netzbreite bereits im Bericht? Erst danach öffnet ihr den Schlusstest.

1. Tragt bei **chosen_hidden_nodes** eure gewählte Breite ein: **16, 128 oder 256**. Ändert nur diese Zahl.
2. Führt E5 aus. Erst jetzt werden die **10'000 Testbilder** ausgewertet.
3. Notiert beide Testquoten und die Änderung im Bericht.
4. Formuliert euren Rat an das Sportgeschäft mit Lernrate, Breite und zwei passenden Zahlen als Beleg.

**Wichtig:** E5 verwendet innerhalb des gewählten Laufs den gespeicherten Zustand mit dem **kleinsten Validierungs-Loss**.

Änderung in Prozentpunkten = Trefferquote des gewählten Modells minus Trefferquote des Grundmodells. **Von 70 % auf 75 % sind es 5 Prozentpunkte.** Ein negatives Ergebnis ist möglich und bleibt eine gültige Beobachtung.

E5 vergleicht mehrere Änderungen zusammen. Wir können daraus nicht sagen, welche einzelne Änderung den Unterschied verursacht hat. Nutzt die Testbilder nicht zur weiteren Auswahl. Bei erneutem Ausführen zeigt E5 die gespeicherten Werte.

In [ ]:
#@title E5 – Auswahl festlegen und erst jetzt testen
# Wähle anhand von E4. Nur diese Zahl ändern: 16, 128 oder 256.
chosen_hidden_nodes = 128 #@param {type:"integer"}

if chosen_hidden_nodes not in width_runs:
    raise ValueError("Wähle eine geprüfte Netzbreite: 16, 128 oder 256.")
selected_run = width_runs[chosen_hidden_nodes]
if selected_run["learning_rate"] != chosen_learning_rate:
    raise RuntimeError("Die Lernrate wurde nach E4 verändert. Führe vor dem Test E4 nochmals mit deiner Wahl aus.")
selection = (selected_run["learning_rate"], selected_run["hidden_nodes"], selected_run["best_epoch"])

if test_result is not None:
    if test_result["selection"] != selection:
        raise RuntimeError("Der Schlusstest ist bereits erfolgt. Beurteile die Ergebnisse; wähle nicht nachträglich am Test aus.")
    print("Bereits geprüfte Auswahl. Die gespeicherten Testwerte werden erneut angezeigt.")
else:
    test_dataset = datasets.FashionMNIST(
        root=str(data_root), train=False, download=True, transform=transform,
    )
    test_loader = make_loader(test_dataset)
    reference_model = FashionMNISTModel(128).to(device)
    reference_model.load_state_dict(baseline_run["first_state"])
    selected_model = FashionMNISTModel(selected_run["hidden_nodes"]).to(device)
    selected_model.load_state_dict(selected_run["best_state"])
    test_result = {
        "selection": selection,
        "reference": evaluate(reference_model, test_loader),
        "selected": evaluate(selected_model, test_loader),
    }
print("Festgelegte Auswahl: Lernrate", selection[0], ", versteckte Neuronen", selection[1],
      ", gespeicherte Epoche", selection[2])
print("Die gespeicherte Epoche wurde mit dem Validierungs-Loss ausgewählt.")
display(pd.DataFrame([
    {"Modell": "Grundmodell nach einer Epoche", "Loss Test": test_result["reference"]["loss"],
     "Richtig": test_result["reference"]["correct"], "Bilder": test_result["reference"]["total"],
     "Treffer Test %": test_result["reference"]["accuracy"]},
    {"Modell": "Gewähltes Modell mit bestem gespeicherten Zustand", "Loss Test": test_result["selected"]["loss"],
     "Richtig": test_result["selected"]["correct"], "Bilder": test_result["selected"]["total"],
     "Treffer Test %": test_result["selected"]["accuracy"]},
]).round(3))
gain = test_result["selected"]["accuracy"] - test_result["reference"]["accuracy"]
print(f"Änderung der Test-Trefferquote: {gain:+.2f} Prozentpunkte")
print("Hier vergleichen wir das Gesamtergebnis mehrerer Änderungen, nicht nur die Lernrate.")
print("Auch eine kleinere Verbesserung, Gleichstand oder Verschlechterung ist eine echte Beobachtung.")

## Minute 45 bis 50 Bericht prüfen und abgeben

- Erklärt einander, weshalb ein kleiner Trainings-Loss allein nicht genügt.
- Prüft die fünf Missionen im gemeinsamen Bericht. Jede Person erklärt einen Teil.
- Stimmen Namen, Werte und die Einheit **Prozentpunkte**?
- Speichert **Colab14_Teamname.docx**. Eine Person lädt die Datei bis Ende der Arbeitszeit in Moodle hoch.

**Pflichtabgabe ist genau ein Beratungsbericht pro Gruppe.** Die fünf belegten Missionen ergeben eure gemeinsame Beraterfreigabe.

**Freiwillig, nur wenn Zeit bleibt:** Weshalb zeigt E5 nicht, welche einzelne Änderung die Verbesserung verursacht hat?

## Technische Reserve mit Übungsdaten

**Nur beim Ersatzweg verwenden. Alle folgenden Zahlen sind didaktisch erstellte Beispielwerte und stammen aus keinem ausgeführten Trainingslauf.** Nutzt entweder überall eigene Messwerte oder überall diese Ersatzdaten. Markiert die Quelle im Bericht. Eure vorher notierte Vermutung bleibt unverändert. Führt dann keine weiteren Trainingszellen aus.

### E1 Training über mehrere Epochen

| Epoche | Loss Training | Loss Validierung | Treffer Validierung |
|---|---:|---:|---:|
| 1 | 0.900 | 0.820 | 71.5 % |
| 6 | 0.300 | 0.440 | 84.0 % |

### E2 Lernraten bei 128 versteckten Neuronen

Notiert eure Vermutung zuerst. Wählt dann nach dem Validierungs-Loss am Ende von sechs Epochen.

| Lernrate | Loss Validierung nach 6 | Treffer Validierung |
|---|---:|---:|
| 0.0001 | 0.580 | 80.0 % |
| 0.001 | 0.440 | 84.0 % |
| 0.01 | 0.500 | 82.0 % |

### E3 Überanpassung im Sonderversuch

512 Trainingsbilder, 256 versteckte Neuronen, Lernrate 0.003, 50 Epochen. Das Minimum des Validierungs-Loss liegt im Beispiel bei Epoche 6. Gezeigt sind ausgewählte Epochen.


| Epoche | Loss Training | Loss Validierung |
|---:|---:|---:|
| 1 | 1.250 | 1.100 |
| 3 | 0.700 | 0.850 |
| 6 | 0.300 | 0.680 |
| 10 | 0.120 | 0.730 |
| 25 | 0.025 | 1.020 |
| 50 | 0.005 | 1.300 |

### E4 Netzbreiten mit Lernrate 0.001

Wieder 6'000 Trainingsbilder, 2'000 Validierungsbilder und sechs Epochen.

| Versteckte Neuronen | Loss Validierung nach 6 | Treffer Validierung | Parameter |
|---:|---:|---:|---:|
| 16 | 0.480 | 83.0 % | 12'730 |
| 128 | 0.440 | 84.0 % | 101'770 |
| 256 | 0.430 | 84.1 % | 203'530 |

**Haltet jetzt eure Auswahl in Mission 4 fest. Lest E5 erst danach.**

### E5 Ersatzdaten zum festgelegten Schlusstest

**Erst nach der Entscheidung in Mission 4 lesen.** Die Auswahl im Beispiel lautet Lernrate 0.001 und 256 versteckte Neuronen. Der beste gespeicherte Zustand stammt aus Epoche 5 mit Validierungs-Loss 0.425. Nach Epoche 6 liegt der Loss bei 0.430.

| Modell | Richtig von 10'000 | Trefferquote Test |
|---|---:|---:|
| Grundmodell nach einer Epoche | 7'120 | 71.20 % |
| Gewähltes Modell aus Epoche 5 | 8'420 | 84.20 % |

Berechnet die Änderung in Prozentpunkten selbst. Schreibt eure Empfehlung und begründet, warum ein kleiner Trainings-Loss allein nicht genügt. Auch diese Testwerte sind didaktische Beispiele und keine gemessenen Ergebnisse.

Copyright 2026 [Markus Ineichen](mailto:markus.ineichen1@sluz.ch)

Code licence: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)